# Current practice as the baseline: what would offering navigation to everyone add?

## At a glance

A hospital program offers transition navigation to some patients going home, and a day-30 survey
asks whether the transition was top box. A quarter of patients never answer. The manager asks what
offering every eligible patient would add to current practice, and what current practice already
achieves over offering no one.

Offering everyone would raise the top-box rate by 0.0956 (95% confidence interval 0.0739 to
0.1173) over the 0.5756 of current practice, which already adds 0.0872 over offering no one. The
usual average effect, 0.1828, compares everyone with no one, so it counts the part current practice
has already collected. With every patient's primary-care visit within 7 days held at yes, the offer
still raises the rate by 0.1153.

The answer assumes that no unrecorded factor drives both the offer and the transition, and that
non-responders would have answered like comparable respondents. On this page's checks the first is
the more fragile: a hidden factor that predicted the transition strongly could erase the gain even
if its link to who got the offer were weaker than that of discharge risk. The data are simulated,
so we know the true gain (0.0799).

## The question

Discharge teams choose who gets the offer, and they favor patients with a high discharge risk. The
manager must decide whether to extend the offer to every eligible patient, so the main quantity is
the gain: the top-box rate if everyone got the offer minus the rate under current practice, over
every eligible patient, whether or not they answer. What current practice achieves is its rate
minus the rate if no one got the offer. Both are population attributable risks (PAR): today's rate
minus the rate if every patient got one option, which the code calls the reference.

A second question asks whether the offer still helps when every patient sees a primary-care doctor
within 7 days. The offer makes that visit more likely, so part of its effect may run through it.
Comparing offer with no offer while every patient's visit is held at yes gives the controlled
direct effect at that level.

```text
  W ────────────────────────────────┐
  │                                 ▼
  ├──────▶ A ──────────────────────▶ Y
  │        │                        ▲
  │        ├──────▶ Z ──────────────┘
  │        │        ▲
  ├────────┼────────┘
  │        ▼
  └──────▶ R          Y is seen only when R = 1

  W = discharge_risk, age, prior_utilization (recorded before the offer)
  A = transition_navigation (the offer)
  Z = pcp_visit_7d (a primary-care visit within 7 days of discharge)
  Y = top_box (the day-30 survey rates the transition top box)
  R = responded (the survey came back by day 45)
  not drawn: an unrecorded cause of A and Y, of Z and Y, or of R and Y,
             and any arrow from Z or Y to R
```

The graph shows what we assume causes what. The characteristics drive the offer, the visit, the
transition, and who answers; the offer drives the visit, the transition, and who answers; and the
visit drives the transition. The analysis rests on what the graph leaves out. Without an arrow from
the transition to answering, non-responders would have answered like respondents with the same
offer and characteristics, an assumption called "missing at random".

## The data

Every fit sets its seeds, so a rerun prints the same numbers.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures

import cleverly

pd.set_option("display.width", 130)
pd.set_option("display.max_columns", 20)
print("cleverly", cleverly.__version__)

cleverly 0.1.6


Another version of `cleverly` can print slightly different numbers.

`make_population_impact` draws 4,000 simulated discharges and also returns the true values, which
real data never give you.

In [2]:
from cleverly.datasets import make_population_impact

# With your own data, load your data frame here, such as frame = pd.read_csv(...), with a
# missing value (NaN) wherever the outcome was not recorded and a 0/1 column that is 1 exactly
# where it was. Real data have no `truth`, so drop every line that uses it.
frame, truth = make_population_impact(n=4_000, seed=2031)
print("rows and columns:", frame.shape)
print(list(frame.columns))
print(frame.head().round(3))
print()
print(f"share offered today:      {frame['transition_navigation'].mean():.3f}")
print(f"share with a 7-day visit: {frame['pcp_visit_7d'].mean():.3f}")
print(f"survey response rate:     {frame['responded'].mean():.3f}")
print()
print("true values, known only because the data are simulated:")
print(f"  top-box rate under current practice: {truth['ey_obs']:.4f}")
print(f"  if every patient got the offer:      {truth['ey1']:.4f}")
print(f"  if no patient got the offer:         {truth['ey0']:.4f}")

rows and columns: (4000, 7)
['top_box', 'transition_navigation', 'discharge_risk', 'age', 'prior_utilization', 'pcp_visit_7d', 'responded']
   top_box  transition_navigation  discharge_risk    age  prior_utilization  pcp_visit_7d  responded
0      1.0                    0.0           0.097  0.445              0.777           1.0        1.0
1      NaN                    0.0           0.308 -0.018             -0.356           1.0        0.0
2      1.0                    1.0           0.691  0.590              0.924           1.0        1.0
3      1.0                    0.0          -1.211  1.138              0.822           1.0        1.0
4      1.0                    0.0          -1.634 -0.696              0.535           1.0        1.0

share offered today:      0.437
share with a 7-day visit: 0.508
survey response rate:     0.758

true values, known only because the data are simulated:
  top-box rate under current practice: 0.5641
  if every patient got the offer:      0.6440
  if no 

Each row is one discharge: `top_box` (1 for top box, `NaN` when the survey did not come back), the
offer, three characteristics measured before it on a standardized scale, the 7-day visit, and
`responded`. The gain the manager asks about is the gap between 0.6440 and 0.5641 in the true
values.

## Why a simple comparison misleads

The quick answers use the respondents: the top-box rate of offered patients against the others,
and the rate among everyone who answered.

In [3]:
answered = frame[frame["responded"] == 1]
rates = answered.groupby("transition_navigation")["top_box"].mean()
print(f"top-box rate among respondents, offered:     {rates[1.0]:.4f}")
print(f"top-box rate among respondents, not offered: {rates[0.0]:.4f}")
print(f"difference:                                  {rates[1.0] - rates[0.0]:.4f}")
print(f"top-box rate among all respondents:          {answered['top_box'].mean():.4f}")
print()
covariates = ["discharge_risk", "age", "prior_utilization"]
print("mean baseline covariate by offer:")
print(frame.groupby("transition_navigation")[covariates].mean().round(3))

top-box rate among respondents, offered:     0.6566
top-box rate among respondents, not offered: 0.5326
difference:                                  0.1241
top-box rate among all respondents:          0.5869

mean baseline covariate by offer:
                       discharge_risk    age  prior_utilization
transition_navigation                                          
0.0                            -0.233  0.115             -0.113
1.0                             0.302 -0.124              0.116


Offered respondents reach a top box 0.1241 more often than the others. That difference answers
neither question. It compares offered with not offered, not everyone with today, and the offered
patients had a higher discharge risk (mean 0.302 against -0.233), which lowers the chance of a top
box. The rate among respondents, 0.5869, overstates today's rate (0.5641 in the true values)
because high-risk patients answer less often.

## Set up the analysis

First, describe the study (`StudyProtocol`). The description does not change the estimate: it
records what the estimate means. The code starts from this program's description and changes four
fields: the population includes patients who never answer, the outcome is the top-box rating, a
death before day 30 counts as not top box, and the reasons add the survey assumption.

In [4]:
from dataclasses import replace

from cleverly.datasets import navigation_protocol

# For your own study, build the description with cleverly.StudyProtocol(...), whose ten fields
# the output prints in order. With missing outcomes, say in target_population that
# non-responders belong to it, and give the reason for missing at random in
# assumption_rationale, as below.
program = navigation_protocol()
protocol = replace(
    program,
    target_population=f"{program.target_population}, whether or not they return the survey",
    outcome=(
        "Top-box patient-reported transition score, from the survey sent on day 30 and "
        "closed on day 45"
    ),
    intercurrent_event_handling=(
        *program.intercurrent_event_handling[:2],
        "The protocol scores death before day 30 as not top box (composite strategy)",
    ),
    assumption_rationale=(
        program.assumption_rationale[0],
        "Given the arm and the baseline variables, survey response carries no further "
        "information about the unobserved score (missingness at random)",
        *program.assumption_rationale[1:],
    ),
)
print("\n".join(protocol.summary_lines()))

Population: Adults with a discharge-home order at a participating hospital during the enrollment period, whether or not they return the survey
Eligibility: Age 18 years or older; Discharge home ordered at a participating hospital
Time zero: Discharge-home order, after baseline measurement and before the navigation offer
Strategies: Offer standard transition navigation; Provide usual discharge support
Versions: Bedside transition plan and two scheduled navigator contacts within 30 days; No access to the transition-navigation offer
Outcome: Top-box patient-reported transition score, from the survey sent on day 30 and closed on day 45
Horizon: 30 days after discharge
Intercurrent events: Use the transition score regardless of readmission; Analyze the offer regardless of completed contacts; The protocol scores death before day 30 as not top box (composite strategy)
Interference unit: Individual patient
Rationale: The recorded baseline variables cover the measured common causes; Given the a

The strategies compare the offer with usual discharge support, and current practice is the mix of
the two that discharge teams chose. These simulated data have no deaths. In real data a death
gives a recorded "not top box" without a survey, so whether an outcome is recorded would depend
on survival, which this fit's response model does not see.

Next, name the columns and the quantity to estimate. `missingness="responded"` marks who answered.
`PopulationAttributableRisk(reference=1)` asks for today's rate minus the rate if every patient got
the offer, so the gain is its negative. `study.identify(...)` lists what must be true for the
answer to be causal.

In [5]:
from cleverly import CausalStudy, PointTreatment, PopulationAttributableRisk

# For your own data, change the column names, the `covariates` list, and the response column.
# The visit comes after the offer, so the population questions leave it out of the data.
design = PointTreatment(
    outcome="top_box",
    treatment="transition_navigation",
    adjustment=tuple(covariates),
    missingness="responded",
)
study = CausalStudy(frame.drop(columns=["pcp_visit_7d"]), design=design, protocol=protocol)
gain_question = study.identify(PopulationAttributableRisk(reference=1))
print(gain_question.summary())

                            Causal question (par)
Question:      mean top_box as          Rows:                            4,000
               treated, minus the mean  Answered by:  adjusting for covariates
               if every unit had
               transition_navigation =
               1
Outcome:       top_box (binary); 3,033
               of 4,000 observed
Treatment:     transition_navigation
               (levels 0, 1)
Adjusted for:  discharge_risk, age,
               prior_utilization
Conditions the answer rests on
condition                       checkable   what it says
------------------------------------------------------------------------------
well-defined treatment          no          Each unit's observed outcome is
                                            the outcome it would have had
                                            under the treatment it actually
                                            received. This needs the treatment
                        

The conditions table is the list to defend. "Well-defined treatment" (an offer always means the
same plan) and "no interference" (one patient's offer does not change another's transition) rest
on the [shared study design](index.md#the-shared-study-design). The data cannot check "no hidden
confounding" or "outcome missing at random". The overlap rows, a real chance of the offer and of
answering for every kind of patient, are probed under "Check the fit". Today's rate alone needs
only the two survey conditions, because it compares no options.

## Estimate current practice and the gain

The method, targeted maximum likelihood estimation (TMLE), fits an outcome model for the chance of
a top box, a treatment model for who gets the offer, and a response model for who answers. A
correction step then weights each respondent by one over their predicted chance of their offer and
of answering, so the respondents stand in for every patient. `NaturalCourseMean()` asks for
today's rate.

- `outcome_learner`: a logistic regression on the offer, the characteristics, and each product of
  two of them, so the offer's effect can differ between patients.
- `treatment_learner`, `missingness_learner`: logistic regressions, which have the true form in
  these simulated data; with real data a flexible learner is the safer default.
- `CrossFitting(n_folds=10)`: each patient's predictions come from models trained without that
  patient, like out-of-fold predictions in cross-validation.
- `Runtime(random_state=2031, n_jobs=1)`: a fixed seed.

In [6]:
from cleverly import CrossFitting, ModelSpec, NaturalCourseMean, Runtime, TMLEMethod


def logistic():
    return LogisticRegression(max_iter=2000)


def with_products():
    return make_pipeline(
        PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), logistic()
    )


# For your own data, swap in your learners.
method = TMLEMethod(
    models=ModelSpec(
        outcome_learner=with_products(),
        treatment_learner=logistic(),
        missingness_learner=logistic(),
    ),
    cross_fitting=CrossFitting(n_folds=10),
    runtime=Runtime(random_state=2031, n_jobs=1),
)
gain_fit = gain_question.estimate(method=method)
current_fit = study.identify(NaturalCourseMean()).estimate(method=method)
print(gain_fit.summary())

                            Effect estimate (TMLE)
Question:      mean top_box as          Rows:                            4,000
               treated, minus the mean  Treated share:                   0.437
               if every unit had        Out-of-fold predictions:      10 folds
               transition_navigation =  Outcome model:     logistic regression
               1                                        after polynomial terms
Outcome:       top_box (binary); 3,033  Treatment model:   logistic regression
               of 4,000 observed        Probability limits:         [0.009532,
Adjusted for:  discharge_risk, age,                                    0.9905]
               prior_utilization        Missingness model:            logistic
Method:        targeted maximum                                     regression
               likelihood (TMLE)
      estimate   std. error             95% CI    p-value
--------------------------------------------------------------------

`par` in the bottom table is today's rate minus the rate with everyone offered, so the gain is
0.096 (interval 0.074 to 0.117). The outcome line, `3,033 of 4,000 observed`, shows that the fit
kept every patient. The interval is valid in large samples when the models are reasonably
accurate.

In [7]:
from cleverly import PopulationAttributableFraction

achieved_fit = study.identify(PopulationAttributableRisk(reference=0)).estimate(method=method)
gain_share_fit = study.identify(PopulationAttributableFraction(reference=1)).estimate(method=method)
achieved_share_fit = study.identify(PopulationAttributableFraction(reference=0)).estimate(
    method=method
)


def line(label, estimate, true, sign=1.0):
    """Print one estimate, its 95% interval, the true value and the support label."""
    low, high = sorted(sign * limit for limit in estimate.ci)
    print(
        f"{label:46s} {sign * estimate.psi:7.4f}  ({low:.4f}, {high:.4f})  "
        f"true {true:.4f}  {estimate.support.label}"
    )


print(f"{'':46s} estimate  95% interval")
line("top-box rate under current practice", current_fit["ey_obs"], truth["ey_obs"])
line("gain if every patient got the offer", gain_fit["par"], -truth["par[reference=1]"], -1.0)
line("  as a share of today's rate", gain_share_fit["paf"], -truth["paf[reference=1]"], -1.0)
line("what current practice achieves over no one", achieved_fit["par"], truth["par[reference=0]"])
line("  as a share of today's rate", achieved_share_fit["paf"], truth["paf[reference=0]"])

                                               estimate  95% interval
top-box rate under current practice             0.5756  (0.5578, 0.5933)  true 0.5641  supported
gain if every patient got the offer             0.0956  (0.0739, 0.1173)  true 0.0799  supported
  as a share of today's rate                    0.1661  (0.1276, 0.2046)  true 0.1416  supported
what current practice achieves over no one      0.0872  (0.0680, 0.1064)  true 0.0850  supported
  as a share of today's rate                    0.1515  (0.1175, 0.1854)  true 0.1507  supported


Offering everyone would lift the top-box rate by 0.0956, a 0.1661 share of today's 0.5756.
Current practice already adds 0.0872 over offering no one, so today's top-box rate would be 0.1515
lower, as a share, without the offer. Each interval contains its true value. The last column,
`supported`, means a simulation study checks this kind of fit; `cleverly` marks a result
`experimental` when none does.

## Why the average effect overstates the gain

A common mistake reports the average treatment effect (ATE), the rate if everyone got the offer
minus the rate if no one did, as the gain from extending the offer. The cell fits it beside the
gain.

In [8]:
from cleverly import ATE

ate_fit = study.identify(ATE(reference=0)).estimate(method=method)
line("average effect, everyone against no one (ATE)", ate_fit["ate"], truth["ate"])
line("gain over current practice", gain_fit["par"], -truth["par[reference=1]"], -1.0)
print()
parts = achieved_fit["par"].psi - gain_fit["par"].psi
not_offered = 1 - frame["transition_navigation"].mean()
print(f"achieved by current practice plus the gain:     {parts:.4f}")
print(f"share not offered today:                        {not_offered:.3f}")
print()
print("true effect of the offer, known only because the data are simulated:")
print(f"  among patients offered today:     {truth['att']:.4f}")
print(f"  among patients not offered today: {truth['atc']:.4f}")

average effect, everyone against no one (ATE)   0.1828  (0.1452, 0.2203)  true 0.1649  supported
gain over current practice                      0.0956  (0.0739, 0.1173)  true 0.0799  supported

achieved by current practice plus the gain:     0.1828
share not offered today:                        0.563

true effect of the offer, known only because the data are simulated:
  among patients offered today:     0.1957
  among patients not offered today: 0.1412


The ATE, 0.1828, splits into what current practice has already collected (0.0872) and what is
left to gain (0.0956), which add up to 0.1828. Its interval, 0.1452 to 0.2203, misses the true gain
of 0.0799. Only the 0.563 of patients not offered today can still gain, and in these simulated data
they gain less than those already offered (true effects 0.1412 against 0.1957): discharge teams
already favor high-risk patients, for whom the offer does more. Reporting the ATE as the gain would
promise the manager the part current practice already delivers.

:::{dropdown} Details: the identity behind the split
With $A$ the offer and $Y^a$ the outcome under option $a$, the gain is
$E(Y^1) - E(Y) = P(A = 0)\,E(Y^1 - Y^0 \mid A = 0)$, the share not offered times the effect
among them (the ATC). What current practice achieves is
$E(Y) - E(Y^0) = P(A = 1)\,E(Y^1 - Y^0 \mid A = 1)$, the share offered times the effect among
them (the ATT), and the two sum to the ATE. The three fits start from the same models, but each
corrects for its own quantity, so the identity need not hold exactly; here it holds to the printed
digits. Each interval comes from its own fit.
:::

## The same comparison as a ratio

Some readers expect a relative effect. `RiskRatio` divides the rate if everyone got the offer by
the rate if no one did, and `OddsRatio` divides the odds.

In [9]:
from cleverly import OddsRatio, RiskRatio

rr_fit = study.identify(RiskRatio(reference=0)).estimate(method=method)
or_fit = study.identify(OddsRatio(reference=0)).estimate(method=method)
line("risk ratio, everyone against no one", rr_fit["rr"], truth["rr"])
line("odds ratio, everyone against no one", or_fit["or"], truth["or"])

risk ratio, everyone against no one             1.3742  (1.2846, 1.4700)  true 1.3442  supported
odds ratio, everyone against no one             2.1379  (1.8207, 2.5104)  true 1.9667  supported


Offering everyone would make a top box 1.3742 times as likely as offering no one (interval 1.2846
to 1.4700). The odds ratio, 2.1379, is further from 1, as it always is when the rate rises, and the
gap is large because a top box is common here; read it as a ratio of odds, not of rates. Both
compare everyone with no one, so neither measures the gain over current practice.

## Does the offer help beyond the 7-day visit?

If the program could guarantee every patient a primary-care visit within 7 days, would the offer
still help? `ControlledDirectEffect(intermediate=1.0)` compares offer with no offer while every
visit is held at yes. `intermediate=` names the visit column, and `intermediate_learner` predicts
it. The fit uses every patient for every model (`CrossFitting(enabled=False)`), the setup its
simulation study checks. Its added conditions include that every kind of patient, offered or not,
had a real chance of a visit, that no unrecorded factor drives both the visit and the transition,
and that answering the survey does not depend on the visit, as the graph assumes.

In [10]:
from cleverly import ControlledDirectEffect

visit_study = CausalStudy(
    frame, design=replace(design, intermediate="pcp_visit_7d"), protocol=protocol
)
held_at_yes = ControlledDirectEffect(intermediate=1.0, contrast=ATE(reference=0))
in_sample = TMLEMethod(
    models=ModelSpec(
        outcome_learner=with_products(),
        treatment_learner=logistic(),
        missingness_learner=logistic(),
        intermediate_learner=logistic(),
    ),
    cross_fitting=CrossFitting(enabled=False),
    runtime=Runtime(random_state=2031, n_jobs=1),
)
direct_fit = visit_study.identify(held_at_yes).estimate(method=in_sample)
line("effect with every visit held at yes", direct_fit["ate"], truth["cde_ate[visit=1]"])
line("effect with the visit left to follow the offer", ate_fit["ate"], truth["ate"])

effect with every visit held at yes             0.1153  (0.0586, 0.1720)  true 0.1160  supported
effect with the visit left to follow the offer  0.1828  (0.1452, 0.2203)  true 0.1649  supported


With every patient's visit held at yes, the offer still raises the top-box rate by 0.1153
(interval 0.0586 to 0.1720), which contains the true 0.1160. So the offer does more than produce
the visit. The ATE beside it lets the visit follow the offer; the two come from different fits, so
the page gives no interval for their difference. Holding the visit at no could give another answer,
so this is not a share of the effect that the visit explains.

## Check the fit

This is the overlap check. The gain predicts every patient's outcome with the offer, so every kind
of patient needs a real chance of the offer and of answering under it. The correction step divides
by those chances, and the fit caps them at 0.0095 for the offer and 0.01 for answering. A model's
calibration slope is 1 when its probabilities are right on average, and below 1 when they are too
extreme.

In [11]:
assessment = gain_fit.assess()
support = assessment.report("support")
nuisance = assessment.report("nuisance_models")
scores = assessment.report("score_equations")
models = {model.name: model.metrics for model in nuisance.models}
attention = tuple(item.name for item in assessment.attention)
answering = support.mechanisms["P(Delta=1|A,W)"]
joint = support.mechanisms["P(A=a,Delta=1|W)"]
print("gain fit, checks that need attention:", attention or "none")
print(f"  smallest chance of an option:                 {support.truncated['most_extreme']:.3f}")
print(f"  smallest chance of answering under an option: {answering['min']:.3f}")
print(f"  smallest chance of an option and answering:   {joint['min']:.4f}")
print(f"  share of patients at the caps:                {joint['clipped_fraction']:.4f}")
for name, label in (("propensity", "offer"), ("missingness", "response"), ("outcome", "outcome")):
    slope, slope_se = models[name]["calibration_slope"], models[name]["calibration_slope_se"]
    print(f"  {label} model calibration slope: {slope:.2f} (standard error {slope_se:.2f})")
print(" ", nuisance.summary().splitlines()[-1])
print(f"  correction step solved its equations: {'PASS' if scores.passed else 'FAIL'}")
print()
direct_assessment = direct_fit.assess()
direct_support = direct_assessment.report("support").mechanisms
direct_attention = tuple(item.name for item in direct_assessment.attention)
print("direct-effect fit, checks that need attention:", direct_attention or "none")
print(f"  smallest chance of a visit:                   {direct_support['P(Z=1|A,W)']['min']:.3f}")
triple = direct_support["P(A=a,Delta=1,Z=z|W)"]
print(f"  smallest chance of an option, answering, and a visit: {triple['min']:.4f}")
print(f"  share of patients at the caps:                {triple['clipped_fraction']:.4f}")

gain fit, checks that need attention: none
  smallest chance of an option:                 0.025
  smallest chance of answering under an option: 0.268
  smallest chance of an option and answering:   0.0249
  share of patients at the caps:                0.0000
  offer model calibration slope: 1.00 (standard error 0.05)
  response model calibration slope: 0.98 (standard error 0.07)
  outcome model calibration slope: 0.95 (standard error 0.07)
  VERDICT: nuisance fits look reasonable.
  correction step solved its equations: PASS

direct-effect fit, checks that need attention: none
  smallest chance of a visit:                   0.119
  smallest chance of an option, answering, and a visit: 0.0045
  share of patients at the caps:                0.0000


Nothing needs attention in either fit, and no patient reached a cap. For the gain, the smallest
chances are 0.025 for the offer and 0.268 for answering, and the calibration slopes sit near 1.
"Nuisance fits look reasonable" means the model checks found no problem, and `PASS` means the
correction step did its job. The direct-effect fit is thinner: one patient's chance of their
offer, an answer, and a visit together is 0.0045, so a few respondents carry large weights, which
widens its interval. None of these checks shows that the models or the estimates are right.

## What if non-responders differ in ways we can't see?

The worry is that offered patients with a poor transition skipped the survey more often, and so
did patients without the offer whose transition went well; either makes the true gain smaller. This
check moves the missing outcomes that way step by step and recomputes the gain without refitting
(`missingness_tilt`). At a given `gamma`, the odds of a top box among non-responders are
`exp(gamma)` times lower than among comparable respondents with the offer, and that many times
higher without it.

In [12]:
from cleverly.sensitivity import missingness_tilt

# For your own data, arm_gamma gives each option's direction: here a positive gamma raises the
# missing outcomes without the offer (0) and lowers those with it (1).
direction = {0: 1.0, 1: -1.0}
curve = missingness_tilt(gain_fit, (0.0, 0.5, 1.0, 1.5, 2.0), arm_gamma=direction).to_frame()
curve["gain"] = -curve["psi"]
curve["odds factor"] = np.exp(curve["gamma"])
print(curve[["gamma", "odds factor", "gain"]].round(4).to_string(index=False))
print()
both = gain_fit.sensitivity.tipping_gamma("par", arm_gamma=direction)
offered_only = gain_fit.sensitivity.tipping_gamma("par", arm_gamma={0: 0.0, 1: -1.0})
print(f"gamma at which the gain reaches 0: {both:.3f} (odds factor {np.exp(both):.2f})")
found = "none in the range searched" if offered_only is None else f"{offered_only:.3f}"
print("the same, moving only the missing outcomes with the offer:", found)

 gamma  odds factor    gain
   0.0       1.0000  0.0956
   0.5       1.6487  0.0682
   1.0       2.7183  0.0414
   1.5       4.4817  0.0176
   2.0       7.3891 -0.0017



gamma at which the gain reaches 0: 1.950 (odds factor 7.03)
the same, moving only the missing outcomes with the offer: none in the range searched


The gain falls as `gamma` grows and reaches zero at 1.950, where the odds of a top box among
non-responders would be 7.03 times lower than among comparable respondents with the offer and 7.03
times higher without it. Moving only the offered non-responders never erases the gain in the range
searched. Whether the survey could miss that strongly in both directions at once is a judgment
about this program.

:::{dropdown} Details: how the outcomes are moved
This is a pattern-mixture analysis. Within each arm and covariate profile, it adds `gamma` times
the arm's multiplier in `arm_gamma` to the logit of the fitted respondents' mean, so with
`{0: 1.0, 1: -1.0}` a positive `gamma` raises the unobserved mean without the offer and lowers it
with the offer. Each row reuses the targeted outcome model of `gamma` = 0 without a new correction
step, so it is a plug-in. The PAR moves through both of its parts, the rate under current
practice and the rate if every patient got the offer. Each value comes from the fitted outcome
model, so it carries that model's error.
:::

## What if a confounder is missing?

Today's rate compares no options, so a hidden confounder cannot bias it, but it can bias the rate
if everyone got the offer. This check bounds that rate (`CounterfactualMean(treatment=1)`) and
subtracts today's. The confounder has two strengths, each a share from 0 to 1: on the outcome side
(`cf_y`), the share of the transition's remaining variation it explains, and on the weight side
(`cf_d`), the share of the variation in the correction weights that only it adds. Each recorded
characteristic serves as a yardstick (`benchmark`, `omitted_confounding`).

In [13]:
from cleverly import CounterfactualMean

everyone_fit = study.identify(CounterfactualMean(treatment=1)).estimate(method=method)
today = current_fit["ey_obs"].psi
sensitivity = everyone_fit.sensitivity
print(f"rate if everyone got the offer: {everyone_fit['ey1'].psi:.4f}, today: {today:.4f}")

# For your own data, benchmark against every recorded confounder.
benchmarks = {}
for covariate in covariates:
    record = sensitivity.benchmark(covariates=(covariate,))
    bound = sensitivity.omitted_confounding(benchmark=record)
    benchmarks[covariate] = (record, bound)
    print()
    print(f"a confounder as strong as {covariate}:")
    print(f"  outcome-side strength, cf_y  {record.cf_y:.3f}")
    print(f"  weight-side strength, cf_d   {record.cf_d:.3f}")
    print(f"  range of the gain            [{bound.lower - today:.4f}, {bound.upper - today:.4f}]")

# The weight-side strength at which the lowest everyone-offered rate reaches today's rate, for a
# confounder that explains all the remaining outcome variation (cf_y = 1), found by bisection.
low, high = 0.0, 0.99
for _ in range(40):
    middle = (low + high) / 2
    if sensitivity.omitted_confounding(cf_y=1.0, cf_d=middle).lower > today:
        low = middle
    else:
        high = middle
tipping_cf_d = (low + high) / 2
print()
print(f"weight-side strength that erases the gain at cf_y = 1: {tipping_cf_d:.3f}")

rate if everyone got the offer: 0.6711, today: 0.5756



a confounder as strong as discharge_risk:
  outcome-side strength, cf_y  0.041
  weight-side strength, cf_d   0.047
  range of the gain            [0.0623, 0.1289]



a confounder as strong as age:
  outcome-side strength, cf_y  0.000
  weight-side strength, cf_d   0.000
  range of the gain            [0.0956, 0.0956]



a confounder as strong as prior_utilization:
  outcome-side strength, cf_y  0.012
  weight-side strength, cf_d   0.008
  range of the gain            [0.0882, 0.1030]



weight-side strength that erases the gain at cf_y = 1: 0.016


A confounder as strong as `discharge_risk`, the strongest recorded one, would leave the gain
between 0.0623 and 0.1289. The recorded characteristics explain little of the transition (their
outcome-side strengths are 0.041 or less), so a hidden factor could be stronger on that side. A
confounder that explains all the remaining variation would need a weight-side strength of only
0.016 to erase the gain, below the 0.047 of `discharge_risk`. Only the program staff can judge
whether such a factor exists.

:::{dropdown} Details: what the strengths mean
`cf_y` is the share of the outcome variation, left after the offer and the recorded
characteristics, that a missing confounder explains. `cf_d` is the share of the second moment of
the Riesz representer that only the missing confounder adds. For the rate if everyone got the
offer, the representer combines the inverse offer probability and the inverse response
probability. The bound uses `rho = 1`, the worst case, and it reads the point estimate only, so
the gain's range here ignores sampling error. `benchmark()` refits without one covariate and
reports gain statistics that serve as proxies for these strengths. These simulated data have no
hidden confounder, so the bounds are stress tests.
[Omitted-variable bounds](../technical-reference/validation-methods.md#omitted-variable-bounds-robustness-value-benchmark-and-contours)
defines each quantity.
:::

## What we can and can't conclude

For the manager: offering navigation to every eligible patient would raise the 30-day top-box rate
by about 0.096 (95% interval 0.074 to 0.117) over current practice, counting patients who never
answer the survey. Current practice already adds about 0.087 over offering no one, which is why
the everyone-against-no-one effect of 0.183 is the wrong number for this decision. A guaranteed
7-day visit would not replace the offer, which still helps with every visit held at yes.

These answers rest on assumptions the data cannot check. Non-responders must have answered like
comparable respondents; erasing the gain would take a departure with an odds factor of 7.03 in both
arms at once. No unrecorded factor may drive both the offer and the transition, and one that
predicts the transition well could erase the gain. The direct effect needs the same for the visit.
The page shows one simulated sample with a known answer, which does not show how the methods
perform in general.

## Go deeper

### How we know the method works

Simulation studies check each fit on other simulated data with outcomes missing at random, a
single characteristic with three levels, and tree or exact models:

| section | study | what it uses |
| --- | --- | --- |
| today's rate | [stacked missing-outcome natural-course CV-TMLE](../technical-reference/method-evidence/stacked-missing-outcome-natural-course-cvtmle.md) | ten folds, as here |
| the gain and what current practice achieves, with their shares | [missing-outcome attributable effects](../technical-reference/method-evidence/missing-outcome-attributable-effects-tmle.md) | ten folds in one scenario, as here |
| the ATE, the ratios, and the rate under the offer | [stacked arm-indexed missing-outcome CV-TMLE](../technical-reference/method-evidence/stacked-arm-indexed-missing-outcome-cvtmle.md) | ten folds, as here |
| the effect with the visit held at yes | [controlled direct-effect TMLE](../technical-reference/method-evidence/controlled-direct-effect-tmle.md) | in sample, as here, against R `tmle` 2.1.1 |

No study checks these fits with this page's logistic models on continuous characteristics. The
direct-effect study gives both implementations the exact model predictions, so it does not check
models fitted from data. The bounds of "What if a confounder is missing?" and the checks of "Try to
break the result" read descriptive numbers, because their studies cover fits without missing
outcomes or fits in sample. The
[technical entry](../technical-reference/point-treatment-tmle.md#validation-issues-special-to-this-method)
lists the methods' evidence and limits.

### Try to break the result

These checks change the data in a known way and refit 20 times each. They run on the ATE fit,
because `cleverly` runs them on a comparison of the two options:

- `placebo` shuffles the offer, so the estimate should be about zero;
- `random_common_cause` adds a column of noise, so the estimate should barely move;
- `subset` refits on random 70% samples, so the estimates should spread about as much as the
  standard error implies for 70% subsamples of these data (0.0126 here).

In [14]:
from cleverly.validation import refute

refutation = refute(ate_fit, tests=("placebo", "subset", "random_common_cause"))
columns = ["test", "original", "refuted_mean", "refuted_sd", "expectation", "verdict"]
print(refutation.to_frame()[columns].round(4).to_string(index=False))

               test  original  refuted_mean  refuted_sd       expectation     verdict
            placebo    0.1828        0.0015      0.0197               ~ 0 descriptive
             subset    0.1828        0.1873      0.0114 scatter ~ 0.01256 descriptive
random_common_cause    0.1828        0.1827      0.0009          ~ 0.1828 descriptive


All three behave as expected: the shuffled offer gives an estimate near zero, the noise column
barely moves it, and the 70% refits spread about as much as the standard error predicts, as far as
twenty refits can tell. These checks test the method, so they cannot detect a hidden confounder or
a survey that is not missing at random. The `verdict` column reads `descriptive`: `cleverly` gives
a pass or fail only when a simulation study backs the check for this kind of fit, and none covers a
fit with missing outcomes.

### Test the method on an outcome with a known effect

A stronger test keeps the real patients, their offers, and who answered, replaces the top-box
outcome with a simulated one whose true effect we know, and checks whether the method recovers
it. A missing outcome stays missing. `dummy_outcome` simulates an outcome that nothing affects, so
its true effect is 0. `simulated_outcome` simulates one that depends on the characteristics and
the offer. `plasmode` also resamples the patients and redraws their offers and answers from models
fitted once to these data. `BernoulliAdjustmentOutcome` draws a 0/1 outcome, and each check runs
10 refits, the fewest `refute` accepts.

In [15]:
from cleverly.validation import BernoulliAdjustmentOutcome

# Simulated outcomes with a known effect. For your own data, set effect to an effect size you
# care about and adjustment_scale to how strongly the characteristics move the outcome, both on
# the log-odds scale; effect=0.0 with adjustment_scale=0.0 is an outcome nothing affects.
independent = BernoulliAdjustmentOutcome(effect=0.0, adjustment_scale=0.0)
adjusted = BernoulliAdjustmentOutcome()
generated = {
    "dummy_outcome": {"dummy_outcome": independent},
    "simulated_outcome": {"simulated_outcome": adjusted},
    "plasmode": {"simulated_outcome": adjusted},
}
generated_tests = {}
for name, declaration in generated.items():
    report = refute(ate_fit, tests=(name,), n_replicates=10, **declaration)
    check = generated_tests[name] = report.tests[0]
    bias = check.readings[0]
    print(
        f"{name:18s} {check.n_replicates} draws, level {check.rung}: "
        f"true value {check.declared_effect:.4f}, mean estimate {check.mean:.4f}, "
        f"{bias.name} {bias.estimate:.3f} in ({bias.lower:.2f}, {bias.upper:.2f})"
    )

dummy_outcome      10 draws, level L1: true value 0.0000, mean estimate 0.0081, bias 0.624 in (0.00, 1.24)


simulated_outcome  10 draws, level L1: true value 0.2040, mean estimate 0.2054, bias 0.084 in (-0.54, 0.70)


plasmode           10 draws, level L1: true value 0.2040, mean estimate 0.2074, bias 0.189 in (-0.43, 0.81)


The mean estimates land near their true values: 0.0081 against 0 for `dummy_outcome`, and close to
0.2040 for the other two. The bias is printed in units of the spread of the refit estimates, with a
95% interval. Ten refits print level `L1`, the numbers without a pass or fail, because the
tolerance band needs at least 400 refits. With ten, an interval such as that of `dummy_outcome`,
whose lower end sits at 0.00, says little.

### How strong would a confounder be on a risk-ratio scale?

The E-value asks how strongly a missing confounder would need to be linked to both the offer and
the top-box outcome, as a risk ratio, to explain the risk ratio away. Epidemiology papers often
report it.

In [16]:
evalue = rr_fit.sensitivity.evalue()
print(evalue.summary())

                                   E-value
Question:  how strong a confounder      Estimate:                           rr
           would explain away rr        Scale:                      risk ratio
                             risk ratio    E-value
------------------------------------------------------------------------------
estimate                          1.374       2.09
95% limit nearest the null        1.285       1.89
An unmeasured confounder would need a risk ratio of 2.09 with both the
treatment and the outcome, beyond the measured covariates, to explain away the
estimate; 1.89 to move the interval to the null.
Notes:
[1] The E-value is the strength at which the confounder's associations with
    the treatment and with the outcome, taken as equal, could explain the
    estimate away. Unequal strengths can trade off: one association may be
    below the E-value if the other is above it.


A missing confounder would need a risk ratio of 2.09 with both the offer and the outcome, beyond
the recorded characteristics, to explain the risk ratio of 1.374 away, and 1.89 to move its
interval to 1. A weaker link to one can be offset by a stronger link to the other. The E-value is
about everyone against no one; it is not a bound on the gain over current practice.

### Two checks this page cannot run

A negative-control check needs an outcome that the offer cannot change and that the program records
before the offer. These data have none. The hidden-confounder simulation (`simulated_confounding`)
flips some recorded offers. Here the offer changes who answers, so a flipped offer would break
missing at random, and `cleverly` refuses the simulation unless the response depends on the
characteristics alone. Roadmap item
[F12](../roadmap.md#f12-missing-outcome-simulated-confounding-replay) would lift that limit.

## Where to go next

The [survey non-response example](survey-nonresponse.ipynb) treats unreturned surveys in more
depth, with a continuous score and the everyone-against-no-one question. The technical entry's
sections on [population interventions](../technical-reference/point-treatment-tmle.md#population-interventions)
and [missing outcomes and controlled direct effects](../technical-reference/point-treatment-tmle.md#missing-outcomes-and-controlled-direct-effects)
give the exact estimators. The [examples index](index.md) lists every example.